# QueryChat - `on_tool_request` Experimentation

In this notebook I'll be experimenting with the `on_tool_request` command to experiment intercepting LLM tool calls to validate, log, or transform them before execution. 

## Experiment 1: Log tool calls

This experiment is mostly to check what tool the LLM requests and what arguments it sends. For this experiment I will print both the **tool name** and the **arguments**. This could be useful to serve as a baseline and to unedrstand how querychat behaves **before** actually changing anything.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) # add project root to path

from dotenv import load_dotenv
from querychat import QueryChat
from chatlas import ChatAnthropic
import os
import pandas as pd
from src.data import load_dashboard_data
import re

In [2]:
# Experiment 1: Log tool requests with on_tool_request
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Load the same dataset used in the dashboard
df = load_dashboard_data()

# Reuse the same greeting and data description from app.py
ai_greeting = """
👋 Hi! I'm your AI burnout explorer.

Ask me questions about employee burnout, productivity, AI usage, workload, and work-life balance.

Examples:
- Show employees with high burnout risk
- Show employees with high AI usage and low productivity
- Sort employees by burnout risk from highest to lowest
- Which job roles have the highest burnout risk?
- Show employees with high manual work hours

You can also say Reset to clear the current AI filter/sort.
"""

ai_data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- focus_hours_per_day: average focus/deep work hours per day.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- burnout_risk_score: numeric burnout risk score.
- burnout_risk_level: burnout category label.
- productivity_score: numeric productivity score.
- work_life_balance_score: numeric work-life balance score.
- workload_score: derived workload metric combining manual work, meetings, and deadline pressure.
- workload_band: workload category (Low, Medium, High).
- ai_band: AI usage category (Low, Moderate, High).

This dataset can be used to analyze:
- Burnout risk by role or experience
- AI usage patterns across employees
- Links between productivity and burnout
- Work-life balance differences
- Manual work and deadline pressure patterns
- High-risk employee subgroups
"""

In [3]:
# Store logs here
tool_logs = []
current_prompt = None

# Callback for on_tool_request
def log_tool_request(req):
    global current_prompt

    print("\n--- TOOL REQUEST ---")
    print("Prompt:", current_prompt)
    print("Tool name:", req.name)
    print("Arguments:", req.arguments)

    tool_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": req.name,
            "arguments": str(req.arguments),
        }
    )

# Create QueryChat client
client = ChatAnthropic(model="claude-sonnet-4-0")

# Create QueryChat object
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

# Attach the callback to the client used by QueryChat
client = qc.client()
client.on_tool_request(log_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [4]:
'''
test_prompts = [
    "Show employees with high burnout risk",
    "Show employees with high AI usage and low productivity",
    "Sort employees by burnout risk from highest to lowest",
    "Which job roles have the highest burnout risk?",
    "Show employees with high manual work hours"
]
'''
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
    "Group the dataset by job_role and compute average burnout_risk_score"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>



```python
# 🔧 tool request (toolu_01FMfJVQiJEak82E6LbPDSAX)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80", title="High burnout risk employees (score > 80)")
```




```python
# ✅ tool result (toolu_01FMfJVQiJEak82E6LbPDSAX)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I notice that this filter returned no results. Looking at the schema, the `burnout_risk_score` has a range of 3.22 to 10.0, so there are no employees with scores above 80. 

The burnout risk score appears to be on a scale of 1-10. Would you like to try a different threshold? For example:

* <span class="suggestion">Show employees with burnout_risk_score > 8</span>
* <span class="suggestion">Filter to employees with burnout_risk_level = 'High'</span>
* <span class="suggestion">What's the average burnout risk score across all employees?</span>


--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'High burnout risk employees (score > 80)'}

FINAL RESPONSE:
I notice that this filter returned no results. Looking at the schema, the `burnout_risk_score` has a range of 3.22 to 10.0, so there are no employees with scores above 80. 

The burnout risk score appears to be on a scale of 1-10. Would you like to try a different threshold? For example:

* <span class="suggestion">Show employees with burnout_risk_score > 8</span>
* <span class="suggestion">Filter to employees with burnout_risk_level = 'High'</span>
* <span class="suggestion">What's the average burnout risk score across all employees?</span>

USER PROMPT: Show employees where manual_work_hours_per_week > 40


<br>



```python
# 🔧 tool request (toolu_01EX5XNuXnNincJKDLGAXmnk)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with high manual work hours (> 40 per week)")
```




```python
# ✅ tool result (toolu_01EX5XNuXnNincJKDLGAXmnk)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees who spend more than 40 hours per week on manual work. You can explore these high-workload employees further:

* <span class="suggestion">What's the average burnout risk score for these high manual work employees?</span>
* <span class="suggestion">Show only employees with manual_work_hours_per_week > 40 AND burnout_risk_level = 'High'</span>
* <span class="suggestion">Sort by burnout_risk_score descending</span>
* <span class="suggestion">How many employees have manual work hours above 40?</span>


--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with high manual work hours (> 40 per week)'}

FINAL RESPONSE:
The dashboard now shows employees who spend more than 40 hours per week on manual work. You can explore these high-workload employees further:

* <span class="suggestion">What's the average burnout risk score for these high manual work employees?</span>
* <span class="suggestion">Show only employees with manual_work_hours_per_week > 40 AND burnout_risk_level = 'High'</span>
* <span class="suggestion">Sort by burnout_risk_score descending</span>
* <span class="suggestion">How many employees have manual work hours above 40?</span>

USER PROMPT: Sort the dataset by burnout_risk_score descending


<br>



```python
# 🔧 tool request (toolu_01Wk4Lt1FKUxUrs57ZKcuEU4)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC", title="High manual work employees sorted by burnout risk (highest first)")
```




```python
# ✅ tool result (toolu_01Wk4Lt1FKUxUrs57ZKcuEU4)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees with high manual work hours (> 40 per week) sorted by their burnout risk score from highest to lowest. This lets you see which high-workload employees are at the greatest risk of burnout.

You can continue exploring these patterns:

* <span class="suggestion">What's the correlation between manual work hours and burnout risk?</span>
* <span class="suggestion">Show the top 10 employees with highest burnout risk scores</span>
* <span class="suggestion">Filter to only show employees with burnout_risk_level = 'High'</span>
* <span class="suggestion">Compare productivity scores for high vs low manual work employees</span>


--- TOOL REQUEST ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'High manual work employees sorted by burnout risk (highest first)'}

FINAL RESPONSE:
The dashboard now shows employees with high manual work hours (> 40 per week) sorted by their burnout risk score from highest to lowest. This lets you see which high-workload employees are at the greatest risk of burnout.

You can continue exploring these patterns:

* <span class="suggestion">What's the correlation between manual work hours and burnout risk?</span>
* <span class="suggestion">Show the top 10 employees with highest burnout risk scores</span>
* <span class="suggestion">Filter to only show employees with burnout_risk_level = 'High'</span>
* <span class="suggestion">Compare productivity scores for high vs low manual work employees</span

<br>



```python
# 🔧 tool request (toolu_01WYWfthymnDuwKsZVe4e95p)
querychat_query(query="SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC", _intent="Calculate average burnout risk score by job role")
```




```python
# ✅ tool result (toolu_01WYWfthymnDuwKsZVe4e95p)
[ { 'job_role': 'Manager',
    'avg_burnout_risk_score': 8.950659509202453,
    'employee_count': 652},
  { 'job_role': 'Developer',
    'avg_burnout_risk_score': 8.371219730941707,
    'employee_count': 1115},
  { 'job_role': 'Writer',
    'avg_burnout_risk_score': 8.347809110629074,
    'employee_count': 461},
  { 'job_role': 'Marketer',
    'avg_burnout_risk_score': 8.28904255319149,
    'employee_count': 658},
  { 'job_role': 'Analyst',
    'avg_burnout_risk_score': 8.23448430493274,
    'employee_count': 892},
  { 'job_role': 'Designer',
    'avg_burnout_risk_score': 8.004487534626037,
    'employee_count': 722}]
```

<br>

Here are the average burnout risk scores by job role:

| Job Role | Avg Burnout Risk Score | Employee Count |
|----------|----------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Managers have the highest average burnout risk** at 8.95, followed by Developers at 8.37. Designers have the lowest average burnout risk at 8.00, though all roles show relatively high burnout scores (8+ on a 10-point scale).

You might want to explore further:

* <span class="suggestion">Show only Manager employees with burnout_risk_level = 'High'</span>
* <span class="suggestion">Compare AI tool usage hours across different job roles</span>
* <span class="suggestion">What factors contribute most to Manager burnout risk?</span>
* <span class="suggestion">Filter to employees with job_role = 'Manager' and manual_work_hours_per_week > 35</span>


--- TOOL REQUEST ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Arguments: {'query': 'SELECT \n    job_role,\n    AVG(burnout_risk_score) AS avg_burnout_risk_score,\n    COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}

FINAL RESPONSE:
Here are the average burnout risk scores by job role:

| Job Role | Avg Burnout Risk Score | Employee Count |
|----------|----------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Managers have the highest average burnout risk** at 8.95, followed by Developers at 8.37. Designers have the lowest average burnout risk at 8.00, though all roles show relatively high burnout scores (8+ on a 10-point scale).

You might want to explore

In [5]:
log_df = pd.DataFrame(tool_logs)
pd.set_option("display.max_colwidth", None)
log_df

,prompt,tool_name,arguments
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'High burnout risk employees (score > 80)'}"
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with high manual work hours (> 40 per week)'}"
2,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'High manual work employees sorted by burnout risk (highest first)'}"
3,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': 'SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}"


### Experiment 1 - Results
This experiment confirms that `on_tool_request` successfully intercepts tool calls, different prompts trigger different tools, and that LLM translates the user's requests into SQL queries. **Filtering and sorting prompts** triggered `querychat_update_dashboard` since it applies SQL filters to the dataset. While **aggregation prompts** triggered the `querychat_query` since it runs SQL queries to compute summaries.

## Experiment 2: Validate tool calls (Updated)

Taking into consideration the feedback received, in this experiment instead of treating every underscore word as a dataset column, I’ll make the validation a bit smarter:

- allow real dataframe columns
- allow SQL aliases introduced with AS
- flag only suspicious column-like names that are neither real columns nor aliases

This updated approach should make the validation more realistic and directly addresses the feedback of the experiment being too simple. Additionally, this rule was chosen because it is still lightweight enough for experimentation, but it better reflects how QueryChat generates aggregation queries. It provides a more meaningful validation step by checking schema consistency without incorrectly flagging legitimate derived fields such as `employee_count` or `avg_burnout_risk_score`.

In [26]:
# Reject requests that mention columns outside our dataset

# Reset logs
validation_logs = []
current_prompt = None

# Get valid column names from the dataframe
valid_columns = set(df.columns)

def validate_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments
    query_text = arguments.get("query", "")
    query_lower = query_text.lower()
    
    # 1. Capture aliases introduced with AS alias_name
    alias_matches = re.findall(r"\bas\s+([a-zA-Z_][a-zA-Z0-9_]*)", query_lower)
    aliases = set(alias_matches)

    # 2. Capture column-like tokens
    tokens = re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", query_lower)

    # 3. Keep only dataframe-style names (underscore names are the most relevant here)
    candidate_columns = {token for token in tokens if "_" in token}

    # 4. Remove aliases from the candidates
    source_like_columns = candidate_columns - aliases

    # 5. Find invalid source-column references
    invalid_columns = sorted(source_like_columns - valid_columns)
    is_valid = len(invalid_columns) == 0

    print("\n--- VALIDATION CHECK (ALIAS-AWARE) ---")
    print("Prompt:", current_prompt)
    print("Tool name:", tool_name)
    print("Aliases:", sorted(aliases))
    print("Candidate columns:", sorted(candidate_columns))
    print("Source-like columns:", sorted(source_like_columns))
    print("Invalid columns:", invalid_columns)
    print("Valid request:", is_valid)

    validation_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "arguments": str(arguments),
            "aliases": str(sorted(aliases)),
            "candidate_columns": str(sorted(candidate_columns)),
            "source_like_columns": str(sorted(source_like_columns)),
            "invalid_columns": str(invalid_columns),
            "is_valid": is_valid,
        }
    )
    

In [27]:
client = ChatAnthropic(model="claude-sonnet-4-0")
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)
client = qc.client()
client.on_tool_request(validate_tool_request)

# "Filter the dataset to employees with burnout_score > 80" is the "bad" prompt
test_prompts = [
    #"Filter the dataset to employees with burnout_risk_score > 80",
    #"Show employees where manual_work_hours_per_week > 40",
    #"Group the dataset by job_role and compute average burnout_risk_score",
    "For each job_role, show employee count and average productivity_score",
    "Filter the dataset to employees with high stress_score",
    "Use stress_score to show high-risk employees",
    "Query the table where overtime_hours_per_week > 10",
    "Use stress_score to show high-risk employees",
    "Query the table where overtime_hours_per_week > 10"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: For each job_role, show employee count and average productivity_score


<br>



```python
# 🔧 tool request (toolu_01Brk7KP1dT3UPYznUgzvrAA)
querychat_query(query="SELECT 
    job_role,
    COUNT(*) AS employee_count,
    ROUND(AVG(productivity_score), 1) AS avg_productivity_score
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_productivity_score DESC", _intent="Show employee count and average productivity score by job role")
```




```python
# ✅ tool result (toolu_01Brk7KP1dT3UPYznUgzvrAA)
[ { 'job_role': 'Developer',
    'employee_count': 1115,
    'avg_productivity_score': 71.4},
  {'job_role': 'Writer', 'employee_count': 461, 'avg_productivity_score': 71.3},
  { 'job_role': 'Marketer',
    'employee_count': 658,
    'avg_productivity_score': 66.1},
  { 'job_role': 'Analyst',
    'employee_count': 892,
    'avg_productivity_score': 65.5},
  { 'job_role': 'Designer',
    'employee_count': 722,
    'avg_productivity_score': 61.5},
  { 'job_role': 'Manager',
    'employee_count': 652,
    'avg_productivity_score': 51.3}]
```

<br>

Here's the breakdown by job role:

| Job Role | Employee Count | Average Productivity Score |
|----------|----------------|---------------------------|
| Developer | 1,115 | 71.4 |
| Writer | 461 | 71.3 |
| Marketer | 658 | 66.1 |
| Analyst | 892 | 65.5 |
| Designer | 722 | 61.5 |
| Manager | 652 | 51.3 |

**Key insights:**
- **Developers** have both the largest employee count (1,115) and highest average productivity (71.4)
- **Writers** achieve nearly identical productivity to developers (71.3) despite being a much smaller group (461)
- **Managers** show significantly lower productivity scores (51.3) compared to other roles
- There's a notable productivity gap between the top performers (Developers/Writers ~71) and bottom performer (Managers ~51)

You might want to explore:
* <span class="suggestion">Show me managers with high productivity scores</span>
* <span class="suggestion">What's the burnout risk level distribution by job role?</span>
* <span class="suggestion">How does AI tool usage vary across job roles?</span>


--- VALIDATION CHECK (ALIAS-AWARE) ---
Prompt: For each job_role, show employee count and average productivity_score
Tool name: querychat_query
Aliases: ['avg_productivity_score', 'employee_count']
Candidate columns: ['avg_productivity_score', 'employee_count', 'job_role', 'productivity_score']
Source-like columns: ['job_role', 'productivity_score']
Invalid columns: []
Valid request: True

FINAL RESPONSE:
Here's the breakdown by job role:

| Job Role | Employee Count | Average Productivity Score |
|----------|----------------|---------------------------|
| Developer | 1,115 | 71.4 |
| Writer | 461 | 71.3 |
| Marketer | 658 | 66.1 |
| Analyst | 892 | 65.5 |
| Designer | 722 | 61.5 |
| Manager | 652 | 51.3 |

**Key insights:**
- **Developers** have both the largest employee count (1,115) and highest average productivity (71.4)
- **Writers** achieve nearly identical productivity to developers (71.3) despite being a much smaller group (461)
- **Managers** show significantly lower producti

<br>

I don't see a column called "stress_score" in this dataset. The available stress/burnout related columns are:

- **burnout_risk_score** (3.22 to 10.0)
- **burnout_risk_level** (High, Medium, Low)
- **deadline_pressure_level** (High, Medium, Low)

Did you mean one of these instead? For example:
* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show employees with high deadline pressure level</span>


FINAL RESPONSE:
I don't see a column called "stress_score" in this dataset. The available stress/burnout related columns are:

- **burnout_risk_score** (3.22 to 10.0)
- **burnout_risk_level** (High, Medium, Low)
- **deadline_pressure_level** (High, Medium, Low)

Did you mean one of these instead? For example:
* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show employees with high deadline pressure level</span>

USER PROMPT: Use stress_score to show high-risk employees


...


FINAL RESPONSE:
I don't see a column called "overtime_hours_per_week" in this dataset. The available work-related time columns are:

- **ai_tool_usage_hours_per_week** (0.0 to 30.4)
- **manual_work_hours_per_week** (10.0 to 43.3)
- **learning_time_hours_per_week** (0.0 to 11.5)
- **meeting_hours_per_week** (1.0 to 30.2)
- **collaboration_hours_per_week** (2.0 to 20.0)
- **focus_hours_per_day** (1.0 to 8.0)

Would you like to filter by one of these columns instead? For example:

* <span class="suggestion">Show employees with manual work hours per week > 30</span>
* <span class="suggestion">Filter to employees with meeting hours per week > 20</span>
* <span class="suggestion">Show employees with high AI tool usage hours per week</span>
* <span class="suggestion">What are all the available columns in this dataset?</span>

USER PROMPT: Use stress_score to show high-risk employees


<br>

I understand you're looking for high-risk employees, but there isn't a "stress_score" column in this dataset. However, I can help you identify high-risk employees using the burnout and risk metrics that are available.

The closest equivalent to what you're looking for would be the **burnout_risk_score** or **burnout_risk_level** columns. Would you like me to:

* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show the top 10% of employees by burnout risk score</span>
* <span class="suggestion">What's the distribution of burnout risk levels?</span>

These will help you identify the employees who are at highest risk for burnout and workplace stress.


FINAL RESPONSE:
I understand you're looking for high-risk employees, but there isn't a "stress_score" column in this dataset. However, I can help you identify high-risk employees using the burnout and risk metrics that are available.

The closest equivalent to what you're looking for would be the **burnout_risk_score** or **burnout_risk_level** columns. Would you like me to:

* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show the top 10% of employees by burnout risk score</span>
* <span class="suggestion">What's the distribution of burnout risk levels?</span>

These will help you identify the employees who are at highest risk for burnout and workplace stress.

USER PROMPT: Query the table where overtime_hours_per_week > 10


<br>

I don't see a column called "overtime_hours_per_week" in this dataset. The available work-hour related columns are:

- **manual_work_hours_per_week** (10.0 to 43.3 hours)
- **ai_tool_usage_hours_per_week** (0.0 to 30.4 hours)
- **meeting_hours_per_week** (1.0 to 30.2 hours)
- **collaboration_hours_per_week** (2.0 to 20.0 hours)
- **learning_time_hours_per_week** (0.0 to 11.5 hours)
- **focus_hours_per_day** (1.0 to 8.0 hours)

Perhaps you meant one of these? I can filter the data using any of the actual columns:

* <span class="suggestion">Show employees with manual work hours per week > 35</span>
* <span class="suggestion">Filter to employees with meeting hours per week > 20</span>
* <span class="suggestion">Show employees with high workload score</span>
* <span class="suggestion">What columns are available in this dataset?</span>


FINAL RESPONSE:
I don't see a column called "overtime_hours_per_week" in this dataset. The available work-hour related columns are:

- **manual_work_hours_per_week** (10.0 to 43.3 hours)
- **ai_tool_usage_hours_per_week** (0.0 to 30.4 hours)
- **meeting_hours_per_week** (1.0 to 30.2 hours)
- **collaboration_hours_per_week** (2.0 to 20.0 hours)
- **learning_time_hours_per_week** (0.0 to 11.5 hours)
- **focus_hours_per_day** (1.0 to 8.0 hours)

Perhaps you meant one of these? I can filter the data using any of the actual columns:

* <span class="suggestion">Show employees with manual work hours per week > 35</span>
* <span class="suggestion">Filter to employees with meeting hours per week > 20</span>
* <span class="suggestion">Show employees with high workload score</span>
* <span class="suggestion">What columns are available in this dataset?</span>


In [28]:
validation_df = pd.DataFrame(validation_logs)
pd.set_option("display.max_colwidth", None)
validation_df

,prompt,tool_name,arguments,aliases,candidate_columns,source_like_columns,invalid_columns,is_valid
0,"For each job_role, show employee count and average productivity_score",querychat_query,"{'query': 'SELECT \n job_role,\n COUNT(*) AS employee_count,\n ROUND(AVG(productivity_score), 1) AS avg_productivity_score\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_productivity_score DESC', '_intent': 'Show employee count and average productivity score by job role'}","['avg_productivity_score', 'employee_count']","['avg_productivity_score', 'employee_count', 'job_role', 'productivity_score']","['job_role', 'productivity_score']",[],True


### Experiment 2 - Results

The alias-aware validation rule successfully distinguished between real dataset columns and SQL aliases generated in aggregation queries. Filtering queries referencing valid fields such as `burnout_risk_score` and `manual_work_hours_per_week` were correctly validated, while aggregation queries using derived aliases like `avg_productivity_score` and `employee_count` were no longer incorrectly flagged as invalid. In several cases, prompts containing non-existent fields (e.g., `stress_score`) did not appear in the validation logs because the LLM identified the schema issue before issuing a tool request and suggested alternative valid columns instead. This suggests that schema awareness can occur both at the model reasoning stage and through the `on_tool_request` validation layer, which provides an additional safeguard for queries that reach the execution stage.

## Experiment 3: Transform tool calls

In this experiment I intercept the tool request and adjust it slightly before execution. Several potential transformations were considered, like:

- cleaning the query title to standardize them
- normalizing column names
- modifying the SQL query structure

For this experiment, I focused only on **cleaning and standardizing the query title** because it modifies metadata associated with the tool request without changing the underlying SQL query or dataset results. Since the dashboard relies on the exact query output for KPIs, visualizations, and CSV downloads, modifying the query itself could unintentionally alter the results shown in the application.

Transformations such as normalizing column names or modifying query structure were not implemented because they would require deeper parsing of the generated SQL and could interfere with QueryChat’s internal query generation. The experiment demonstrates how `on_tool_request` can safely adjust tool arguments before execution while preserving the integrity of the underlying data operations.

In [9]:
# standardize title
transform_logs = []
current_prompt = None

def transform_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments

    original_title = arguments.get("title", None)

    # Transform the title if it exists
    if original_title:
        new_title = f"LLM Result: {original_title}"
        arguments["title"] = new_title
    else:
        new_title = None

    print("\n--- TRANSFORMATION ---")
    print("Prompt:", current_prompt)
    print("Tool:", tool_name)
    print("Original title:", original_title)
    print("New title:", new_title)

    transform_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "original_title": original_title,
            "new_title": new_title,
        }
    )

In [10]:
client.on_tool_request(transform_tool_request)

test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
]

for prompt in test_prompts:
    current_prompt = prompt

    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>



```python
# 🔧 tool request (toolu_018cfYvifRY36WyVjyMhFifN)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80", title="Employees with burnout risk score > 80")
```




```python
# ✅ tool result (toolu_018cfYvifRY36WyVjyMhFifN)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

This filter returned no results again because the `burnout_risk_score` ranges from 3.22 to 10.0, so no employees have scores above 80. The scale appears to be 1-10, not 1-100.

For high burnout risk employees, consider these options instead:

* <span class="suggestion">Show employees with burnout_risk_score > 9</span>
* <span class="suggestion">Filter to employees with burnout_risk_level = 'High'</span>  
* <span class="suggestion">Show employees with burnout_risk_score >= 8.5</span>
* <span class="suggestion">What's the highest burnout risk score in the dataset?</span>


--- TRANSFORMATION ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool: querychat_update_dashboard
Original title: Employees with burnout risk score > 80
New title: LLM Result: Employees with burnout risk score > 80

--- VALIDATION CHECK ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Candidate columns: ['burnout_risk_score']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'LLM Result: Employees with burnout risk score > 80'}

FINAL RESPONSE:
This filter returned no results again because the `burnout_risk_score` ranges from 3.22 to 10.0, so no employees have scores above 80. The scale appears to be 1-10, not 1-100.

For high burnout risk employees, consider these options instead:



<br>



```python
# 🔧 tool request (toolu_01MTfo3zGwF8C8LtVPenbXAN)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with high manual work hours (> 40 per week)")
```




```python
# ✅ tool result (toolu_01MTfo3zGwF8C8LtVPenbXAN)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees with more than 40 hours of manual work per week. As we saw earlier, this is a small group of only 11 employees with extremely high workloads and burnout risk.

You can explore these high-workload employees:

* <span class="suggestion">Sort by burnout_risk_score descending</span>
* <span class="suggestion">Which job roles are represented in this high manual work group?</span>
* <span class="suggestion">How do their productivity scores compare to the overall average?</span>
* <span class="suggestion">Show their work-life balance scores</span>


--- TRANSFORMATION ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool: querychat_update_dashboard
Original title: Employees with high manual work hours (> 40 per week)
New title: LLM Result: Employees with high manual work hours (> 40 per week)

--- VALIDATION CHECK ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Candidate columns: ['manual_work_hours_per_week']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'LLM Result: Employees with high manual work hours (> 40 per week)'}

FINAL RESPONSE:
The dashboard now shows employees with more than 40 hours of manual work per week. As we saw earlier, this is a small group of only 11 employees with extremely high workloads and burnout risk.

You can expl

<br>



```python
# 🔧 tool request (toolu_013tJ1jQaBMVuidGFLEoQ5Yy)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC", title="High manual work employees sorted by burnout risk (highest first)")
```




```python
# ✅ tool result (toolu_013tJ1jQaBMVuidGFLEoQ5Yy)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows the high manual work employees (> 40 hours per week) sorted by burnout risk score from highest to lowest. This reveals which of these already high-workload employees are at the greatest risk.

You can continue analyzing these extreme cases:

* <span class="suggestion">What are the specific burnout risk scores for these top employees?</span>
* <span class="suggestion">Compare their AI tool usage hours to see if automation could help</span>
* <span class="suggestion">Show their work-life balance and productivity scores</span>
* <span class="suggestion">Reset the dashboard to explore other employee groups</span>


--- TRANSFORMATION ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool: querychat_update_dashboard
Original title: High manual work employees sorted by burnout risk (highest first)
New title: LLM Result: High manual work employees sorted by burnout risk (highest first)

--- VALIDATION CHECK ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Candidate columns: ['burnout_risk_score', 'manual_work_hours_per_week']
Invalid columns: []
Valid request: True

--- TOOL REQUEST ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'LLM Result: High manual work employees sorted by burnout risk (highest first)'}

FINAL RESPONSE:
The dashboard now shows the high manual work employees (> 40 hours per week) sorted by burnout risk score from highest to lo

In [11]:
transform_df = pd.DataFrame(transform_logs)
pd.set_option("display.max_colwidth", None)
transform_df

,prompt,tool_name,original_title,new_title
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,Employees with burnout risk score > 80,LLM Result: Employees with burnout risk score > 80
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,Employees with high manual work hours (> 40 per week),LLM Result: Employees with high manual work hours (> 40 per week)
2,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,High manual work employees sorted by burnout risk (highest first),LLM Result: High manual work employees sorted by burnout risk (highest first)


### Experiment 3 - Results

In this experiment, `on_tool_request` was used to modify tool arguments before execution. Specifically, the title generated by the LLM for dashboard updates was standardized by adding a prefix (`LLM Result:`). This transformation does not alter the underlying SQL query or dataset results, but demonstrates how intercepted tool calls can be programmatically adjusted before execution.

This experiment shows that `on_tool_request` can be used not only for logging and validation, but also for lightweight transformations that improve consistency in the application interface.